# CliniScan+ 02: Annotation Validation & Quality Control
Validates converted YOLO `.txt` labels against original radiographs, visualizes consensus bounding boxes, and checks normalization integrity.

In [ ]:
import os
import glob
import cv2
import matplotlib.pyplot as plt
from PIL import Image

PROCESSED_DIR = r'../data/processed'
val_images = glob.glob(os.path.join(PROCESSED_DIR, 'images', 'val', '*.jpg'))
print(f'Total validation images: {len(val_images)}')

# Inspect first 4 validation samples
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
for idx, img_path in enumerate(val_images[:4]):
    img_name = os.path.basename(img_path)
    lbl_path = os.path.join(PROCESSED_DIR, 'labels', 'val', img_name.replace('.jpg', '.txt'))
    img = cv2.imread(img_path)
    h, w = img.shape[:2]
    
    if os.path.exists(lbl_path):
        with open(lbl_path) as lf:
            lines = lf.readlines()
        for line in lines:
            parts = line.strip().split()
            if len(parts) == 5:
                cls_id, xc, yc, bw, bh = int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                x1, y1 = int((xc - bw/2)*w), int((yc - bh/2)*h)
                x2, y2 = int((xc + bw/2)*w), int((yc + bh/2)*h)
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 4)
                cv2.putText(img, f'Class {cls_id}', (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    
    ax = axes[idx // 2, idx % 2]
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f'Sample: {img_name[:12]}...')
    ax.axis('off')
plt.tight_layout()
plt.show()